# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import pandas
import dask.distributed
import geopandas

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling
import plotting

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:52311/status,
Dashboard: http://127.0.0.1:52311/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:52312,Workers: 0
Dashboard: http://127.0.0.1:52311/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:52335,Total threads: 2
Dashboard: http://127.0.0.1:52336/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:52315,


# Values to edit

In [3]:
max_cloud_covers = [15]#, 1, 10, 0] # percentage
lowtide_search_range = 60 # days - +/- this devided by 2

In [4]:
site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [5]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
survey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"
mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"

In [14]:
for max_cloud_cover in max_cloud_covers:
    print(f"Max cloud percentage {max_cloud_cover}")
    for site in site_names:
        print(site)
        polygon_file = utils.get_site_polygon_path(site)
        uav_polygon = geopandas.read_file(polygon_file)
        
        satellite_file = utils.get_training_satellite_path(site_name=site, 
                                                           use_threshold=False,
                                                           max_cloud_cover=max_cloud_cover,
                                                           filtered=False,
                                                           )
        if satellite_file.exists():
            continue

        for i in range(2):
            try:
                satellite_data = sentinel2.get_cloud_filtered_images_near_date(
                    site_name=site,
                    geometry=uav_polygon,
                    max_cloud_cover=max_cloud_cover,
                    date_file=survey_dates_file,
                    low_tide_search_days=lowtide_search_range,
                )
                if len(satellite_data['time']) == 0:
                    print("Warning: No satellite data within the specified low tide and cloud cover and anomaly filter constraints. Ignore")
                else:
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                    satellite_data = satellite_data.rio.clip(uav_polygon.geometry, drop=True, all_touched=True)
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    utils.save_netcdf(satellite_data, satellite_file)
                    break
            except Exception:
                print("Caught exception downloading images and try again.")
        print(f"Save RGB for {site}")
        satellite_images_path = utils.get_training_satellite_path(
            site_name=site,
            use_threshold=True, 
            max_cloud_cover=max_cloud_cover,
            filtered=True)
        utils.rgb_from_satellite(satellite_data_path=satellite_images_path)

Max cloud percentage 15
Ihutai
CatlinsLake
CatlinsRiverMouth
Childrens
Duvauchelle
	Satellite date range 2025-01-30/2025-03-31
	Low Cloud Tiles: 10 from the low tide tiles of 22. Cloud percentages [100.  100.    0.    0.   57.6  99.8   0.    0.   27.2   0.1 100.    0.
 100.    2.1 100.   95.9 100.    0.2 100.    0.    0.   38. ] Kept [False False  True  True False False  True  True False  True False  True
 False  True False False False  True False  True  True False]
Save RGB for Duvauchelle
Satellite data path C:\Local\repos\seagrass-detection\data\training\tide_thresholded_cloud_15_filtered\satellite_images\Duvauchelle_sentinel-2.nc does not exist.
Robinsons
	Satellite date range 2025-01-31/2025-04-01
	Low Cloud Tiles: 11 from the low tide tiles of 22. Cloud percentages [100.  100.    0.    0.   98.6 100.    0.    0.   50.4   0.  100.    0.
 100.    8.6 100.    0.  100.   25.9 100.    0.    0.    0.2] Kept [False False  True  True False False  True  True False  True False  True
 False

In [15]:
satellite_path = satellite_file.parent
mean_mndwi_file = satellite_path / "mean_mndwi.csv"
tide_time_file = satellite_path / "tide_time.csv"

In [16]:
mean_mndwi_all_sites = []

for site in site_names:
    mean_mndwi_series = []
    print(site)
    
    satellite_file = satellite_path / f"{site}_sentinel-2.nc"
    if not satellite_file.exists():
        print(f"WARNING no {satellite_file.name}.")
        continue

    # load in satellite file
    satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
    # calculate MNDWI layer
    mean_mndwi_series = sentinel2._mean_mndwi_by_date(satellite_data).compute().to_pandas()

    mean_mndwi_all_sites.append(mean_mndwi_series)

# for plotting the gallery / distributions
mean_mndwi_all_sites = pandas.concat(mean_mndwi_all_sites, keys=site_names, names=["site", "time"]).rename("mean_mndwi")
mean_mndwi_all_sites.to_csv(mean_mndwi_file)

Ihutai
CatlinsLake
CatlinsRiverMouth
Childrens
Duvauchelle
Robinsons
Takamatua
Purau
IveyBay_Nov25
IveyBay_Feb26
LeftBank_Nov25
LeftBank_Feb26
Paremata_Nov25
Paremata_Feb26
Paremata_Feb25
ThePoint_Nov25
ThePoint_Feb26
Takapuwahia_Nov25
Takapuwahia_Feb26
IveyBay_ThePoint_LeftBank_Oct24


In [17]:
mean_mndwi = pandas.read_csv(mean_mndwi_file)
output_folder = satellite_path /  "rbg_thumbnails_mndwi_sorted"
output_folder.mkdir(exist_ok=True)

for site in site_names:
    print(f"{site}")
    gallery_file = output_folder / f"{site}.html"
    if gallery_file.exists():
        continue
    plotting.create_rgb_gallery(site_name=site,
                                sort_value=mean_mndwi, 
                                sort_value_label="mean_mndwi",
                                gallery_file=gallery_file, 
                                max_cloud_cover=max_cloud_cover,
                                satellite_path=satellite_path)

Ihutai
CatlinsLake
CatlinsRiverMouth
Childrens
	Site Childrens - create gallery
  Saved C:\Local\repos\seagrass-detection\data\training\cloud_15\satellite_images\rbg_thumbnails_mndwi_sorted\Childrens.html
Duvauchelle
	Site Duvauchelle - create gallery
  Saved C:\Local\repos\seagrass-detection\data\training\cloud_15\satellite_images\rbg_thumbnails_mndwi_sorted\Duvauchelle.html
Robinsons
	Site Robinsons - create gallery
  Saved C:\Local\repos\seagrass-detection\data\training\cloud_15\satellite_images\rbg_thumbnails_mndwi_sorted\Robinsons.html
Takamatua
	Site Takamatua - create gallery
  Saved C:\Local\repos\seagrass-detection\data\training\cloud_15\satellite_images\rbg_thumbnails_mndwi_sorted\Takamatua.html
Purau
	Site Purau - create gallery
  Saved C:\Local\repos\seagrass-detection\data\training\cloud_15\satellite_images\rbg_thumbnails_mndwi_sorted\Purau.html
IveyBay_Nov25
	Site IveyBay_Nov25 - create gallery
  Saved C:\Local\repos\seagrass-detection\data\training\cloud_15\satellite_ima

In [23]:
tide_times_all_sites = []
site_names_included = []
for site in site_names:
    print(site)
    tide_info = []

    site_polygon_file = utils.get_site_polygon_path(site)
    site_polygon = geopandas.read_file(site_polygon_file)
    centroid_nzgd49 = site_polygon.to_crs(utils.CRS_NZGD49).geometry.union_all().centroid
    latitude_nzgd49, longitude_nzgd49 = float(centroid_nzgd49.y), float(centroid_nzgd49.x)
    
    satellite_file = satellite_path / f"{site}_sentinel-2.nc"
    if not satellite_file.exists():
        print(f"WARNING no {satellite_file.name}.")
        continue
    site_names_included.append(site)

    # load in satellite file
    satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
    # calculate distances from low tide
    timestamps = satellite_data["time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").values
    for timestamp in timestamps:
        tide_dict = sentinel2._get_low_tide_info_for_datetime(timestamp, latitude_nzgd49, longitude_nzgd49)
        tide_dict["time"] = timestamp
        tide_info.append(tide_dict)
    tide_time_series.append(pandas.DataFrame(tide_info))

# for plotting the gallery / distributions
tide_times_all_sites = pandas.concat(tide_time_series, keys=site_names_included, names=["site", "date id"])
tide_times_all_sites.to_csv(tide_time_file)

Ihutai
CatlinsLake
CatlinsRiverMouth
Childrens
Duvauchelle
Robinsons
Takamatua
Purau
IveyBay_Nov25
IveyBay_Feb26
LeftBank_Nov25
LeftBank_Feb26
Paremata_Nov25
Paremata_Feb26
Paremata_Feb25
ThePoint_Nov25
ThePoint_Feb26
Takapuwahia_Nov25
Takapuwahia_Feb26
IveyBay_ThePoint_LeftBank_Oct24


ValueError: The length of the keys (20) must match the length of the objects to concatenate (41)

In [ ]:
tide_times_all_sites = pandas.read_csv(tide_time_file)
tide_times_all_sites["time"] = pandas.to_datetime(tide_times_all_sites["time"], utc=True)
tide_times_all_sites["minutes_from_low_tide"] = pandas.to_timedelta(tide_times_all_sites["time_from_low_tide"]).dt.total_seconds()/60
output_folder = satellite_path /  "rbg_thumbnails_tide_sorted"
output_folder.mkdir(exist_ok=True)

for site in sites:
    print(f"{site}")
    gallery_file = output_folder / f"{site}.html"
    if gallery_file.exists():
        continue
    plotting.create_rgb_gallery(site_name=site,
                                sort_value=tide_times_all_sites, 
                                sort_value_label="minutes_from_low_tide",
                                gallery_file=gallery_file, 
                                years=years,
                                max_cloud_cover=max_cloud_cover,
                                satellite_path=satellite_path)